# CycleGAN Training on Google Colab

This notebook trains a CycleGAN model for Summer ↔ Winter image translation.

**Critical Setup:**
- Mounts Google Drive for data persistence
- Saves checkpoints to Drive (survives runtime disconnections)
- Uses mixed precision training for speed

**Runtime:** GPU (T4 or better recommended)

## 1. Mount Google Drive & Setup Paths

## 3. Copy Data from Drive to Colab Local Disk (CRITICAL for Speed)

**Why:** Reading images from Drive during training is ~10x slower than local disk.
**Strategy:** Copy zip from Drive → Unzip to Colab local disk → Train fast
**Trade-off:** Uses ~2GB of Colab runtime disk (acceptable, runtime has ~100GB)

In [1]:
import os
import shutil

# 1. Define paths
drive_zip_path = '/content/drive/MyDrive/SeasonsGAN/data/summer2winter_yosemite.zip'
local_extract_path = '/content/data/summer2winter_yosemite'

# 2. Check if already extracted (avoid re-doing if kernel restarts)
if not os.path.exists(local_extract_path):
    print("📦 Copying zip from Drive to Colab local disk... (This makes training 10x faster)")
    
    # Copy zip to local
    shutil.copy(drive_zip_path, '/content/temp.zip')
    print("✓ Zip copied")
    
    # Unzip
    print("📂 Unzipping...")
    !unzip -q /content/temp.zip -d /content/data
    
    # Clean up zip
    os.remove('/content/temp.zip')
    
    print("✅ Done! Data is ready on fast local disk.")
else:
    print("✓ Data already extracted on local disk.")

# 3. Update DATA_PATH to point to local disk (not Drive)
DATA_PATH = local_extract_path
print(f"\n📍 Training will use: {DATA_PATH}")

📦 Copying zip from Drive to Colab local disk... (This makes training 10x faster)


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/SeasonsGAN/data/summer2winter_yosemite.zip'

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define paths (ADJUST 'SeasonsGAN' if you named your Drive folder differently)
PROJECT_PATH = '/content/drive/MyDrive/SeasonsGAN'
DATA_PATH = f'{PROJECT_PATH}/data/summer2winter_yosemite'
CHECKPOINT_PATH = f'{PROJECT_PATH}/checkpoints'

# Create checkpoint directory
os.makedirs(CHECKPOINT_PATH, exist_ok=True)

print(f"✓ Project path: {PROJECT_PATH}")
print(f"✓ Data path: {DATA_PATH}")
print(f"✓ Checkpoint path: {CHECKPOINT_PATH}")

## 3. Install Dependencies

In [ ]:
!pip install -q albumentations tqdm
print("✓ Dependencies installed")

## 4. Upload Project Code to Colab

**Option A:** Upload the `src/` folder manually via Colab UI  
**Option B:** Clone from GitHub (if you push your code there)  
**Option C:** Copy from Drive (if you uploaded the code to Drive)

For now, we'll assume you've uploaded `src/` to `/content/src/`

In [ ]:
# If you uploaded src/ to Drive, copy it to Colab runtime
!cp -r {PROJECT_PATH}/src /content/src

# Add src to Python path
import sys
sys.path.insert(0, '/content/src')

print("✓ Project code loaded")

## 5. Verify Data Structure

Expected structure:
```
summer2winter_yosemite/
├── trainA/  (summer images)
└── trainB/  (winter images)
```

In [ ]:
import os

train_a = f"{DATA_PATH}/trainA"
train_b = f"{DATA_PATH}/trainB"

if os.path.exists(train_a) and os.path.exists(train_b):
    num_summer = len(os.listdir(train_a))
    num_winter = len(os.listdir(train_b))
    print(f"✓ Found {num_summer} summer images")
    print(f"✓ Found {num_winter} winter images")
else:
    print("❌ ERROR: Data not found! Check your DATA_PATH.")

## 6. Update Config with Drive Paths

In [ ]:
from utils import config

# Override config paths to use Drive
config.TRAIN_DIR = DATA_PATH
config.CHECKPOINT_GEN_S = f"{CHECKPOINT_PATH}/gen_summer.pth.tar"
config.CHECKPOINT_GEN_W = f"{CHECKPOINT_PATH}/gen_winter.pth.tar"
config.CHECKPOINT_DISC_S = f"{CHECKPOINT_PATH}/disc_summer.pth.tar"
config.CHECKPOINT_DISC_W = f"{CHECKPOINT_PATH}/disc_winter.pth.tar"

# Print config
config.print_config()

## 7. Test Model Architectures

In [ ]:
from models.generator import test_generator
from models.discriminator import test_discriminator

print("Testing Generator...")
test_generator()

print("\nTesting Discriminator...")
test_discriminator()

## 8. Start Training

**WARNING:** This will run for `NUM_EPOCHS` (default: 200).  
Checkpoints are saved every 5 epochs to your Drive.

In [ ]:
from training import main

# Start training
main()

## 9. Test Inference (After Training)

Transform a test image using the trained generator.

In [ ]:
from inference import load_generator, transform_image
import torch
from PIL import Image
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load generator (Summer -> Winter)
gen_w = load_generator(config.CHECKPOINT_GEN_W, device)

# Transform a test image
test_image_path = f"{DATA_PATH}/trainA/2010-12-01 13_09_18.jpg"  # Adjust to an actual image
output = transform_image(test_image_path, gen_w, device)

# Display
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(Image.open(test_image_path))
axes[0].set_title("Original (Summer)")
axes[0].axis('off')

axes[1].imshow(output)
axes[1].set_title("Generated (Winter)")
axes[1].axis('off')

plt.tight_layout()
plt.show()